# 🧠 Project 14: Low-Resource Text Classification using Data Augmentation
### UG Student Project | Google Colab Version (GPU-Accelerated)

**How to use:**
1. Go to `Runtime → Change runtime type → T4 GPU` ✅
2. Run all cells top to bottom (`Runtime → Run all`)
3. Everything saves to your Colab session — download what you need at the end

---


## ⚙️ Step 0 — Check GPU & Install Packages

In [ ]:
import subprocess, sys

# Check GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU is available!")
    print(result.stdout.split("\n")[8])
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Install required packages (takes ~60 seconds on Colab)
!pip install -q datasets transformers scikit-learn nltk seaborn pandas numpy matplotlib scipy
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✅ All packages installed")

## 📦 Step 1 — Imports & Reproducibility Seeds

In [ ]:
import os, random, copy, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import pairwise_distances
from scipy.stats import entropy as scipy_entropy
from collections import Counter
from datasets import load_dataset
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet, stopwords as nltk_stopwords

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)
os.makedirs("data", exist_ok=True)
os.makedirs("results/week1", exist_ok=True)
os.makedirs("results/week3", exist_ok=True)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEEDS = [42, 123, 7]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {DEVICE}")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

---
## 📅 WEEK 1 — Dataset Analysis & Preprocessing

Loads AG News + IMDb, simulates low-resource setting, splits data, plots class balance.


In [ ]:
STOP_WORDS = set(nltk_stopwords.words("english"))

def preprocess_text(text):
    tokens = word_tokenize(str(text).lower())
    tokens = [t for t in tokens if t.isalpha() and t not in STOP_WORDS]
    return " ".join(tokens)

def simulate_low_resource(df, fraction=0.05, seed=42):
    set_seed(seed)
    return df.groupby("label", group_keys=False).apply(
        lambda x: x.sample(frac=fraction, random_state=seed)
    ).reset_index(drop=True)

def make_splits(df, seed=42):
    train, temp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=seed)
    val, test   = train_test_split(temp, test_size=0.50, stratify=temp["label"], random_state=seed)
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)

print("Loading datasets...")

# ── AG News ───────────────────────────────────────────────────────────────────
# Try new namespaced path first, fall back to direct CSV download
try:
    ag_raw  = load_dataset("fancyzhx/ag_news")
    ag_full = pd.DataFrame(ag_raw["train"])
    print("✅ AG News loaded via HuggingFace")
except Exception:
    print("⚠️  HF load failed, downloading AG News CSV directly...")
    import urllib.request, io, gzip
    url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
    urllib.request.urlretrieve(url, "/tmp/ag_train.csv")
    ag_full = pd.read_csv("/tmp/ag_train.csv", header=None, names=["label","title","description"])
    ag_full["text"]  = ag_full["title"] + " " + ag_full["description"]
    ag_full["label"] = ag_full["label"] - 1   # make 0-indexed
    print("✅ AG News loaded via direct CSV download")

ag_label = {0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech"}
ag_full["label_name"] = ag_full["label"].map(ag_label)
ag_low   = simulate_low_resource(ag_full, fraction=0.05, seed=42)
ag_low["clean_text"] = ag_low["text"].apply(preprocess_text)
ag_low["text_len"]   = ag_low["text"].apply(lambda x: len(x.split()))

# ── IMDb ──────────────────────────────────────────────────────────────────────
try:
    imdb_raw  = load_dataset("stanfordnlp/imdb")
    imdb_full = pd.DataFrame(imdb_raw["train"])
    print("✅ IMDb loaded via HuggingFace")
except Exception:
    try:
        imdb_raw  = load_dataset("imdb")
        imdb_full = pd.DataFrame(imdb_raw["train"])
        print("✅ IMDb loaded via HuggingFace (fallback path)")
    except Exception:
        print("⚠️  HF load failed, downloading IMDb CSV directly...")
        url2 = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
        urllib.request.urlretrieve(url2, "/tmp/imdb.csv")
        imdb_full = pd.read_csv("/tmp/imdb.csv")
        imdb_full.rename(columns={"review":"text","sentiment":"label_name"}, inplace=True)
        imdb_full["label"] = (imdb_full["label_name"] == "positive").astype(int)
        print("✅ IMDb loaded via direct CSV download")

imdb_label = {0:"Negative", 1:"Positive"}
imdb_full["label_name"] = imdb_full["label"].map(imdb_label)
imdb_low  = simulate_low_resource(imdb_full, fraction=0.04, seed=42)
imdb_low["clean_text"] = imdb_low["text"].apply(preprocess_text)
imdb_low["text_len"]   = imdb_low["text"].apply(lambda x: len(x.split()))

print(f"\n✅ AG News  low-resource: {len(ag_low)} samples | {ag_low['label'].nunique()} classes")
print(f"✅ IMDb low-resource: {len(imdb_low)} samples | {imdb_low['label'].nunique()} classes")
print("\nAG News class distribution:")
print(ag_low["label_name"].value_counts().to_string())
print("\nIMDb class distribution:")
print(imdb_low["label_name"].value_counts().to_string())

In [ ]:
# ── Splits ────────────────────────────────────────────────────────────────────
train_ag, val_ag, test_ag     = make_splits(ag_low,   seed=42)
train_im, val_im, test_im     = make_splits(imdb_low, seed=42)

train_ag.to_csv("data/agnews_train.csv",  index=False)
val_ag.to_csv("data/agnews_val.csv",      index=False)
test_ag.to_csv("data/agnews_test.csv",    index=False)
train_im.to_csv("data/imdb_train.csv",    index=False)
val_im.to_csv("data/imdb_val.csv",        index=False)
test_im.to_csv("data/imdb_test.csv",      index=False)

print(f"AG News  — Train:{len(train_ag)} | Val:{len(val_ag)} | Test:{len(test_ag)}")
print(f"IMDb     — Train:{len(train_im)} | Val:{len(val_im)} | Test:{len(test_im)}")
print("✅ Splits saved to data/")

In [ ]:
# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Week 1 — Dataset Analysis", fontsize=15, fontweight="bold")

# AG News class balance
counts_ag = ag_low["label_name"].value_counts()
sns.barplot(x=counts_ag.index, y=counts_ag.values, palette="Blues_d", ax=axes[0,0])
axes[0,0].set_title("AG News — Class Distribution (Low-Resource 5%)")
axes[0,0].set_xlabel("Class"); axes[0,0].set_ylabel("Count")
for i, v in enumerate(counts_ag.values): axes[0,0].text(i, v+2, str(v), ha="center")

# IMDb class balance
counts_im = imdb_low["label_name"].value_counts()
sns.barplot(x=counts_im.index, y=counts_im.values, palette="Reds_d", ax=axes[0,1])
axes[0,1].set_title("IMDb — Class Distribution (Low-Resource 4%)")
axes[0,1].set_xlabel("Class"); axes[0,1].set_ylabel("Count")
for i, v in enumerate(counts_im.values): axes[0,1].text(i, v+2, str(v), ha="center")

# AG News text length
ag_low["text_len"].hist(bins=40, ax=axes[1,0], color="steelblue", edgecolor="white")
axes[1,0].set_title("AG News — Text Length Distribution")
axes[1,0].set_xlabel("Words"); axes[1,0].set_ylabel("Count")
axes[1,0].axvline(ag_low["text_len"].mean(), color="red", linestyle="--",
                   label=f'Mean={ag_low["text_len"].mean():.0f}')
axes[1,0].legend()

# IMDb text length
imdb_low["text_len"].hist(bins=40, ax=axes[1,1], color="coral", edgecolor="white")
axes[1,1].set_title("IMDb — Text Length Distribution")
axes[1,1].set_xlabel("Words"); axes[1,1].set_ylabel("Count")
axes[1,1].axvline(imdb_low["text_len"].mean(), color="blue", linestyle="--",
                   label=f'Mean={imdb_low["text_len"].mean():.0f}')
axes[1,1].legend()

plt.tight_layout()
plt.savefig("results/week1/dataset_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to results/week1/dataset_analysis.png")

---
## 📅 WEEK 2 — Baseline Model & Data Augmentation

Defines:
- **TextCNN** (Kim, 2014) — our baseline neural classifier
- **EDA** — 4 augmentation operations (SR, RI, RS, RD)
- **All coreset methods** — Random, K-Center, Gradient, Proposed CBUS
- **Compression functions** — Magnitude pruning, 8-bit & 4-bit quantization


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VOCABULARY
# ═══════════════════════════════════════════════════════════════════
class Vocabulary:
    def __init__(self, max_vocab=30000):
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.max_vocab = max_vocab

    def build(self, texts):
        counter = Counter(w for t in texts for w in t.split())
        for word, _ in counter.most_common(self.max_vocab - 2):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx]  = word

    def encode(self, text, max_len=128):
        tokens = text.split()[:max_len]
        ids    = [self.word2idx.get(t, 1) for t in tokens]
        ids   += [0] * (max_len - len(ids))
        return ids

    def __len__(self): return len(self.word2idx)

# ═══════════════════════════════════════════════════════════════════
# DATASET
# ═══════════════════════════════════════════════════════════════════
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.data   = [vocab.encode(t, max_len) for t in texts]
        self.labels = labels

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        return (torch.tensor(self.data[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

# ═══════════════════════════════════════════════════════════════════
# TextCNN MODEL (Kim, 2014)
# ═══════════════════════════════════════════════════════════════════
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes,
                 filter_sizes=(2,3,4), num_filters=128, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs     = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes])
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        x = [F.relu(c(x)) for c in self.convs]
        x = [F.max_pool1d(c, c.size(2)).squeeze(2) for c in x]
        x = torch.cat(x, dim=1)
        return self.fc(self.dropout(x))

# ═══════════════════════════════════════════════════════════════════
# TRAIN / EVAL
# ═══════════════════════════════════════════════════════════════════
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def evaluate(model, loader, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds  += model(xb).argmax(1).cpu().tolist()
            labels += yb.tolist()
    return accuracy_score(labels, preds), f1_score(labels, preds, average="weighted")

print("✅ Model classes defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EDA AUGMENTATION
# ═══════════════════════════════════════════════════════════════════
def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_"," ")
            if s.lower() != word.lower(): syns.add(s)
    return list(syns)

def synonym_replacement(text, n=1):
    words = text.split()
    cands = [w for w in words if w not in STOP_WORDS and get_synonyms(w)]
    if not cands: return text
    for w in random.sample(cands, min(n, len(cands))):
        syns = get_synonyms(w)
        if syns: words = [random.choice(syns) if x == w else x for x in words]
    return " ".join(words)

def random_deletion(text, p=0.1):
    words = text.split()
    if len(words) == 1: return text
    res = [w for w in words if random.random() > p]
    return " ".join(res) if res else random.choice(words)

def random_swap(text, n=1):
    words = text.split()
    if len(words) < 2: return text
    for _ in range(n):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return " ".join(words)

def random_insertion(text, n=1):
    words = text.split()
    for _ in range(n):
        cands = [w for w in words if get_synonyms(w)]
        if not cands: break
        syns = get_synonyms(random.choice(cands))
        if syns: words.insert(random.randint(0, len(words)), random.choice(syns))
    return " ".join(words)

def augment_eda(text, num_aug=4, alpha=0.1):
    n = max(1, int(alpha * len(text.split())))
    out = []
    for _ in range(num_aug):
        op = random.choice(["sr","rd","rs","ri"])
        if   op == "sr": out.append(synonym_replacement(text, n))
        elif op == "rd": out.append(random_deletion(text, alpha))
        elif op == "rs": out.append(random_swap(text, n))
        else:            out.append(random_insertion(text, n))
    return out

def augment_dataset(df, num_aug=4, alpha=0.1, seed=42):
    set_seed(seed)
    rows = []
    for _, row in df.iterrows():
        rows.append(row)
        for aug in augment_eda(row["clean_text"], num_aug, alpha):
            r = row.copy(); r["clean_text"] = aug; rows.append(r)
    return pd.DataFrame(rows).reset_index(drop=True)

# ── Quick demo ────────────────────────────────────────────────────
sample = "The president signed the new trade agreement today"
print("Original :", sample)
print("SR       :", synonym_replacement(sample))
print("RD       :", random_deletion(sample))
print("RS       :", random_swap(sample))
print("RI       :", random_insertion(sample))
print("\n✅ EDA functions defined")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# COMPRESSION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════
def magnitude_prune(model, ratio):
    model = copy.deepcopy(model)
    all_w = torch.cat([m.weight.data.abs().flatten()
                       for m in model.modules()
                       if isinstance(m, (nn.Conv1d, nn.Linear))])
    threshold = torch.quantile(all_w, ratio)
    for m in model.modules():
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            m.weight.data *= (m.weight.data.abs() > threshold)
    return model

def quantize_8bit(model):
    return torch.quantization.quantize_dynamic(
        copy.deepcopy(model).cpu(), {nn.Linear}, dtype=torch.qint8)

def simulate_4bit(model):
    model = copy.deepcopy(model)
    with torch.no_grad():
        for p in model.parameters():
            mn, mx = p.data.min(), p.data.max()
            scale  = (mx - mn) / 15.0 + 1e-8
            p.data = torch.round((p.data - mn) / scale) * scale + mn
    return model

# ═══════════════════════════════════════════════════════════════════
# CORESET SELECTION METHODS
# ═══════════════════════════════════════════════════════════════════
def random_coreset(df, frac, seed=42):
    set_seed(seed)
    return df.groupby("label", group_keys=False).apply(
        lambda x: x.sample(frac=frac, random_state=seed)
    ).reset_index(drop=True)

def kcenter_coreset(df, frac, seed=42):
    set_seed(seed)
    n = max(1, int(len(df) * frac))
    X = TfidfVectorizer(max_features=512, sublinear_tf=True).fit_transform(
        df["clean_text"].tolist()).toarray()
    sel  = [random.randint(0, len(X)-1)]
    dmin = np.full(len(X), np.inf)
    while len(sel) < n:
        d = pairwise_distances(X[sel[-1]:sel[-1]+1], X, metric="euclidean")[0]
        dmin = np.minimum(dmin, d)
        dmin[sel] = -np.inf
        sel.append(int(np.argmax(dmin)))
    return df.iloc[sel].reset_index(drop=True)

def gradient_coreset(df, frac, vocab, num_classes, device, seed=42):
    set_seed(seed)
    n  = max(1, int(len(df) * frac))
    m  = TextCNN(len(vocab), 100, num_classes).to(device)
    cr = nn.CrossEntropyLoss()
    ld = DataLoader(TextDataset(df["clean_text"].tolist(), df["label"].tolist(), vocab),
                    batch_size=1, shuffle=False)
    norms = []
    m.train()
    for xb, yb in ld:
        xb, yb = xb.to(device), yb.to(device)
        m.zero_grad()
        cr(m(xb), yb).backward()
        norms.append(sum(p.grad.norm().item()**2
                         for p in m.parameters() if p.grad is not None)**0.5)
    return df.iloc[np.argsort(norms)[-n:]].reset_index(drop=True)

def proposed_coreset(df, frac, seed=42):
    """
    CBUS — Class-Balanced Uncertainty Sampling (Proposed Method ⭐)
    Select highest-entropy (most uncertain) samples, enforcing class balance.
    """
    set_seed(seed)
    n_per = max(1, int(len(df) * frac) // df["label"].nunique())
    X     = TfidfVectorizer(max_features=1024, sublinear_tf=True).fit_transform(
        df["clean_text"].tolist())
    y     = df["label"].values
    proba = LogisticRegression(max_iter=200, random_state=seed).fit(X, y).predict_proba(X)
    ents  = np.array([scipy_entropy(p) for p in proba])
    sel   = []
    for label in df["label"].unique():
        idx = np.where(y == label)[0]
        sel.extend(idx[np.argsort(ents[idx])[-n_per:]].tolist())
    return df.iloc[sel].reset_index(drop=True)

print("✅ Compression & coreset functions defined")

---
## 📅 WEEK 3 — Full Experiments (runs on GPU 🚀)

This is the main experiment cell. It runs **every configuration × 3 seeds** and saves results.

**Expected time on Colab T4 GPU: ~8–15 minutes**

Configurations:
- A. Baseline
- B. EDA Augmentation
- C. Pruning (5%, 10%, 25%, 50%, 90%)
- C. Quantization (8-bit, 4-bit)
- D. Coreset (10%, 25%, 50%) × (random, k-center, gradient, CBUS)
- E. Combined: Aug + Compression + Coreset


In [ ]:
def run_config(train_df, val_df, test_df, num_classes,
               augment=False, prune_ratio=None, quant_bits=None,
               coreset_fn=None, coreset_frac=None,
               epochs=10, batch=64, label="exp"):

    accs, f1s = [], []
    for seed in SEEDS:
        set_seed(seed)
        tr = train_df.copy()

        # 1. Coreset
        if coreset_fn and coreset_frac:
            if   coreset_fn == "random":
                tr = random_coreset(tr, coreset_frac, seed)
            elif coreset_fn == "kcenter":
                tr = kcenter_coreset(tr, coreset_frac, seed)
            elif coreset_fn == "gradient":
                vt = Vocabulary(); vt.build(tr["clean_text"].tolist())
                tr = gradient_coreset(tr, coreset_frac, vt, num_classes, DEVICE, seed)
            elif coreset_fn == "proposed":
                tr = proposed_coreset(tr, coreset_frac, seed)

        # 2. Augmentation
        if augment:
            tr = augment_dataset(tr, num_aug=4, alpha=0.1, seed=seed)

        # 3. Build vocab + loaders
        vocab = Vocabulary(); vocab.build(tr["clean_text"].tolist())
        def loader(df, shuf=False):
            ds = TextDataset(df["clean_text"].tolist(), df["label"].tolist(), vocab)
            return DataLoader(ds, batch_size=batch, shuffle=shuf,
                              num_workers=2, pin_memory=True)
        trl, vl, tel = loader(tr, True), loader(val_df), loader(test_df)

        # 4. Train
        model = TextCNN(len(vocab), 100, num_classes).to(DEVICE)
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        crit  = nn.CrossEntropyLoss()
        best_val, best_state = 0, None
        for _ in range(epochs):
            train_epoch(model, trl, opt, crit, DEVICE)
            va, _ = evaluate(model, vl, DEVICE)
            if va > best_val:
                best_val   = va
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state)

        # 5. Compression
        eval_device = DEVICE
        if prune_ratio is not None:
            model = magnitude_prune(model, prune_ratio).to(DEVICE)
        if quant_bits == 8:
            model = quantize_8bit(model)
            eval_device = torch.device("cpu")
            tel = loader(test_df)
        elif quant_bits == 4:
            model = simulate_4bit(model).to(DEVICE)

        acc, f1 = evaluate(model, tel, eval_device)
        accs.append(acc); f1s.append(f1)

    return {
        "label":    label,
        "acc_mean": float(np.mean(accs)),
        "acc_std":  float(np.std(accs)),
        "f1_mean":  float(np.mean(f1s)),
        "f1_std":   float(np.std(f1s)),
    }

print("✅ Experiment runner ready")

In [ ]:
# Load AG News splits (primary dataset for all experiments)
train_df = pd.read_csv("data/agnews_train.csv")
val_df   = pd.read_csv("data/agnews_val.csv")
test_df  = pd.read_csv("data/agnews_test.csv")
NC       = train_df["label"].nunique()

results = []

def run_and_log(label, **kwargs):
    print(f"  Running: {label}")
    r = run_config(train_df, val_df, test_df, NC, label=label, **kwargs)
    results.append(r)
    print(f"  ✓ Acc={r['acc_mean']:.4f}±{r['acc_std']:.4f}  F1={r['f1_mean']:.4f}±{r['f1_std']:.4f}")
    return r

# ── A. Baseline ───────────────────────────────────────────────────────────────
print("\n" + "═"*55)
print("A. BASELINE")
print("═"*55)
run_and_log("Baseline")

# ── B. Augmentation ───────────────────────────────────────────────────────────
print("\n" + "═"*55)
print("B. EDA AUGMENTATION")
print("═"*55)
run_and_log("Aug_EDA", augment=True)

# ── C. Pruning ────────────────────────────────────────────────────────────────
print("\n" + "═"*55)
print("C. MAGNITUDE PRUNING")
print("═"*55)
for r in [0.05, 0.10, 0.25, 0.50, 0.90]:
    run_and_log(f"Prune_{int(r*100)}%", prune_ratio=r)

# ── C. Quantization ───────────────────────────────────────────────────────────
print("\n" + "═"*55)
print("C. QUANTIZATION")
print("═"*55)
run_and_log("Quant_8bit", quant_bits=8)
run_and_log("Quant_4bit", quant_bits=4)

# ── D. Coreset ────────────────────────────────────────────────────────────────
print("\n" + "═"*55)
print("D. CORESET SELECTION")
print("═"*55)
for frac in [0.10, 0.25, 0.50]:
    for method in ["random", "kcenter", "gradient", "proposed"]:
        run_and_log(f"Coreset_{method}_{int(frac*100)}%",
                    coreset_fn=method, coreset_frac=frac)

# ── E. Combined ───────────────────────────────────────────────────────────────
print("\n" + "═"*55)
print("E. COMBINED")
print("═"*55)
run_and_log("Combined_Aug+Prune25+CBUS50",
            augment=True, prune_ratio=0.25, coreset_fn="proposed", coreset_frac=0.50)
run_and_log("Combined_Aug+Quant8+CBUS50",
            augment=True, quant_bits=8, coreset_fn="proposed", coreset_frac=0.50)

# ── Save ──────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
results_df.to_csv("results/week3/all_results.csv", index=False)
print("\n\n✅ ALL EXPERIMENTS DONE — saved to results/week3/all_results.csv")
print(results_df[["label","acc_mean","acc_std","f1_mean","f1_std"]].to_string(index=False))

---
## 📊 Week 3 — Result Plots (for your paper)

In [ ]:
df = pd.read_csv("results/week3/all_results.csv")

def bar_err(subset, title, metric="acc", ax=None):
    owns = ax is None
    if owns: fig, ax = plt.subplots(figsize=(max(7, len(subset)*0.95), 4.5))
    means, stds = subset[f"{metric}_mean"].values, subset[f"{metric}_std"].values
    colors = ["#1a6faf" if i==0 else "#d93b3b" if "Combined" in subset["label"].iloc[i]
              else "#7fb3d3" for i in range(len(subset))]
    bars = ax.bar(range(len(subset)), means, yerr=stds, capsize=4,
                  color=colors, edgecolor="white")
    ax.set_xticks(range(len(subset)))
    ax.set_xticklabels(subset["label"].tolist(), rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Weighted F1" if metric=="f1" else "Accuracy")
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_ylim(max(0, means.min()-0.12), min(1.0, means.max()+0.14))
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+s+0.004,
                f"{m:.3f}", ha="center", fontsize=7.5)
    if owns: plt.tight_layout(); plt.savefig(f"results/week3/plot_{title[:15].replace(' ','_')}.png", dpi=150); plt.show()

# Plot 1 — Pruning
sub = df[df["label"].str.startswith("Baseline") | df["label"].str.startswith("Prune")]
bar_err(sub, "Magnitude Pruning vs Accuracy", "acc")

# Plot 2 — Quantization
sub = df[df["label"].str.startswith("Baseline") | df["label"].str.startswith("Quant")]
bar_err(sub, "Quantization vs Accuracy", "acc")

# Plot 3 — Coreset methods line plot
fig, ax = plt.subplots(figsize=(8,5))
core = df[df["label"].str.startswith("Coreset")]
methods_c = ["random","kcenter","gradient","proposed"]
colors_c  = ["#4DAF4A","#377EB8","#FF7F00","#E41A1C"]
labels_c  = ["Random","K-Center","Gradient","CBUS (Proposed ⭐)"]
for m, c, l in zip(methods_c, colors_c, labels_c):
    sub = core[core["label"].str.contains(m)].copy()
    sub["frac"] = sub["label"].str.extract(r"(\d+)%").astype(int)
    sub = sub.sort_values("frac")
    ax.errorbar(sub["frac"], sub["acc_mean"], yerr=sub["acc_std"],
                label=l, color=c, marker="o", capsize=4, linewidth=2.2, markersize=7)
base_acc = df[df["label"]=="Baseline"]["acc_mean"].values[0]
ax.axhline(base_acc, linestyle="--", color="gray", linewidth=1.5, label="Baseline (full data)")
ax.set_xlabel("Coreset Fraction (%)"); ax.set_ylabel("Test Accuracy")
ax.set_title("Coreset Selection: All Methods vs. Fraction", fontweight="bold")
ax.legend(); plt.tight_layout()
plt.savefig("results/week3/plot_coreset_methods.png", dpi=150)
plt.show()

# Plot 4 — Full heatmap
pivot_labels = ["Baseline","Aug_EDA","Prune_25%","Prune_50%","Quant_8bit","Quant_4bit",
                "Coreset_random_50%","Coreset_kcenter_50%","Coreset_gradient_50%",
                "Coreset_proposed_50%","Combined_Aug+Prune25+CBUS50","Combined_Aug+Quant8+CBUS50"]
sub = df[df["label"].isin(pivot_labels)].set_index("label").reindex(pivot_labels)
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, metric, title in zip(axes, ["acc_mean","f1_mean"], ["Test Accuracy","Weighted F1"]):
    vals = sub[[metric]].values
    im   = ax.imshow(vals, cmap="RdYlGn", vmin=vals.min()-0.02, vmax=vals.max()+0.02, aspect="auto")
    ax.set_yticks(range(len(pivot_labels))); ax.set_yticklabels(pivot_labels, fontsize=8)
    ax.set_xticks([0]); ax.set_xticklabels([title])
    ax.set_title(title, fontweight="bold")
    for i, v in enumerate(vals.flatten()):
        ax.text(0, i, f"{v:.3f}", ha="center", va="center", fontsize=8.5, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.045)
plt.suptitle("Experiment Summary Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig("results/week3/plot_heatmap.png", dpi=150); plt.show()
print("✅ All plots saved to results/week3/")

---
## 📅 WEEK 4 — Generate Your Paper Results Tables

Prints ready-to-paste LaTeX tables for your final paper.


In [ ]:
df = pd.read_csv("results/week3/all_results.csv")

def latex_table(rows, caption, label):
    print(f"\n% ── {caption} ──")
    print("\\begin{table}[h]\n\\centering")
    print("\\begin{tabular}{lcc}\n\\hline")
    print("\\textbf{Configuration} & \\textbf{Accuracy} & \\textbf{Weighted F1} \\\\\n\\hline")
    for _, r in rows.iterrows():
        acc = f"{r['acc_mean']:.3f}$\\pm${r['acc_std']:.3f}"
        f1  = f"{r['f1_mean']:.3f}$\\pm${r['f1_std']:.3f}"
        print(f"{r['label'].replace('_',' ')} & {acc} & {f1} \\\\")
    print(f"\\hline\n\\end{{tabular}}\n\\caption{{{caption}}}\n\\label{{{label}}}\n\\end{{table}}")

print("="*60)
print("COPY THESE INTO YOUR LATEX PAPER")
print("="*60)

latex_table(df[df["label"].isin(["Baseline","Aug_EDA"])],
            "Effect of EDA Augmentation", "tab:aug")
latex_table(df[df["label"].str.startswith("Prune") | (df["label"]=="Baseline")],
            "Effect of Magnitude Pruning", "tab:prune")
latex_table(df[df["label"].isin(["Baseline","Quant_8bit","Quant_4bit"])],
            "Effect of Quantization", "tab:quant")
latex_table(df[df["label"].str.contains("50%") | (df["label"]=="Baseline")],
            "Coreset Selection (50% fraction)", "tab:coreset")
latex_table(df[df["label"].str.startswith("Combined") | (df["label"]=="Baseline")],
            "Combined Configurations", "tab:combined")

---
## 💾 Download Your Results

Run this cell to zip and download everything.


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("project14_results", "zip", ".", "results")
shutil.make_archive("project14_data",    "zip", ".", "data")

files.download("project14_results.zip")
files.download("project14_data.zip")
print("✅ Download started — check your browser downloads folder")